In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import MinMaxScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D2 in response_OUS
data = list(OUS_D2['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D2, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

In [6]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

### COX assumption in Train data

In [7]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.00001)
cph.fit(df, duration_col='DFS', event_col='event_DFS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>
         test_name = proportional_hazard_test

---
                                                       test_statistic    p  -log2(p)
LBP_003_CT                                                       0.00 0.95      0.08
LBP_003_PET                                                      0.00 0.99      0.01
LBP_012_CT                                                       0.01 0.91      0.13
LBP_012_PET                                                      0.00 0.98      0.03
LBP_021_CT                                                       0.01 0.94      0.09
LBP_021_PET                                                      0.02 0.90      0.16
LBP_030_CT                                                       0.00 0.97      0.04
LBP_030_PET       

In [8]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index([], dtype='object')


In [9]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.1)
cph.fit(df, duration_col='DFS', event_col='event_DFS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>
         test_name = proportional_hazard_test

---
                                                       test_statistic      p  -log2(p)
LBP_003_CT                                                       0.20   0.65      0.62
LBP_003_PET                                                      0.43   0.51      0.96
LBP_012_CT                                                       0.19   0.66      0.59
LBP_012_PET                                                      0.18   0.67      0.57
LBP_021_CT                                                       0.94   0.33      1.59
LBP_021_PET                                                      0.01   0.93      0.10
LBP_030_CT                                                       0.00   0.94      0.08
LB

In [10]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['glszm_GrayLevelNonUniformity_PET_c04',
       'glszm_LargeAreaEmphasis_CT_c16',
       'glszm_LargeAreaLowGrayLevelEmphasis_CT_c16',
       'glszm_ZoneVariance_CT_c16', 'ngtdm_Busyness_d_1_PET_b2'],
      dtype='object')


###### penalizer values essentially result in same result 

## Test dataset: MAASTRO 

In [11]:
(MAASTRO_D2['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [12]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [13]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D2['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [14]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D2, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event
0,1,0.765178,0.610062,30.504395,50.002093,52.886671,58.000000,47.759816,58.864251,38067.208333,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,0.776540,0.504616,21.069386,41.753334,44.922155,44.294469,44.147480,48.723711,17870.791667,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,0.697164,0.478604,21.238275,44.375483,48.466483,45.891176,37.656341,48.969378,17426.500000,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,8.83,1.0
3,4,0.574636,0.446059,20.570459,46.115989,42.544095,47.507894,33.837849,55.226805,12463.791667,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,19.73,1.0
4,6,0.633419,0.480378,26.130155,54.394967,66.030296,56.320511,45.398238,67.089492,28300.125000,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,0.882411,0.577884,19.774388,34.218615,41.785165,40.261644,35.805028,43.520110,12822.458333,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,13.27,1.0
95,111,0.535802,0.455642,23.259099,51.046869,51.264022,51.264022,33.600595,52.440442,18368.291667,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,0.716610,0.631485,31.838169,50.417953,46.324939,56.035703,57.070132,57.671483,35384.541667,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,0.665145,0.628338,28.213278,44.901412,50.289164,50.596443,35.777088,51.478151,26180.208333,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [15]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [16]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,shape_MinorAxisLength,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [17]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [18]:
# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [19]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 374)
y_train:  (139,)


In [20]:
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [21]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 376)

In [22]:
# VIF dataframe 
vif_data = pd.DataFrame() 
vif_data["feature"] = X.columns 
  
# calculating VIF for each feature 
vif_data["VIF"] = [variance_inflation_factor(X.values, i) 
                          for i in range(len(X.columns))] 

vif_data[vif_data['VIF'] > 100000]

,feature,VIF
0,shape_Elongation,2.096355e+11
1,shape_Flatness,3.609232e+11
2,shape_LeastAxisLength,2.691126e+12
3,shape_MajorAxisLength,7.524811e+12
4,shape_Maximum2DDiameterColumn,2.124339e+13
...,...,...
369,LBP_111_PET,5.922282e+11
370,LBP_120_PET,3.305758e+11
371,LBP_201_PET,1.508996e+12
372,LBP_210_PET,1.369708e+12


# Standardization

In [23]:
original_X = X.copy()

In [24]:
# Standardize the data 
## Save the column and index 
X_columns = X.columns 
X_index = X.index

# Standardize non-categorical and then concat with the categorical
scaler = MinMaxScaler()  
X_std = scaler.fit_transform(X)
X_std = pd.DataFrame(X_std, columns=X_columns, index=X_index)

In [25]:
# Standardize the numeric part 
X_MAASTRO_column = X_MAASTRO.columns
X_MAASTRO_index = X_MAASTRO.index

X_MAASTRO_std = scaler.transform(X_MAASTRO)

# Change the standardized part into a dataframe 
X_MAASTRO_std = pd.DataFrame(X_MAASTRO_std, columns=X_MAASTRO_column, index=X_MAASTRO_index)

In [26]:
# Saving the data 
X_new = X 
X_new_std = X_std 
MAASTRO_new = X_MAASTRO 
MAASTRO_new_std = X_MAASTRO_std 

# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [27]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 16:03:31,386] A new study created in memory with name: no-name-81a1b89b-2310-4d71-a58a-af9f11c1bc7c


  0%|          | 0/1 [00:00<?, ?it/s]

[W 2024-04-14 16:03:31,827] Trial 0 failed with parameters: {} because of the following error: LinAlgError('Matrix is singular.').
Traceback (most recent call last):
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/optuna/study/_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/64/jkqp6xyx2hj50dmd2pqfm3780000gn/T/ipykernel_37071/1714754791.py", line 62, in objective
    model.fit(X_train_std, y_train)
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/sksurv/linear_model/coxph.py", line 449, in fit
    delta = solve(
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/scipy/linalg/_basic.py", line 220, in solve
    _solve_check(n, info)
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/scipy/linalg/_basic.py", line 29, in _solve_check
    raise LinAlgError('Matrix is singular.')
numpy.linalg.LinAlgError: Matrix is singular.
[W 2024-04-14 16:03:31,832] Trial 0 fai

LinAlgError: Matrix is singular.

In [28]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

ValueError: No trials are completed yet.

In [29]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

ValueError: No trials are completed yet.

#### Test

In [30]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [31]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

LinAlgError: Matrix is singular.

In [32]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [33]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

NameError: name 'c_index' is not defined

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [34]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 16:16:37,901] A new study created in memory with name: no-name-ead7a42c-724d-4eac-86a2-6a313e49da25


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-14 16:16:38,073] A new study created in memory with name: no-name-9ea83c54-b92c-48ad-a307-17d6b54efb30


Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.6124031007751938
Fold 3 C-index: 0.5148936170212766
Fold 4 C-index: 0.6577946768060836
Fold 5 C-index: 0.6158798283261803
[I 2024-04-14 16:16:38,067] Trial 0 finished with value: 0.5957320931913246 and parameters: {}. Best is trial 0 with value: 0.5957320931913246.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.5957320931913246], datetime_start=datetime.datetime(2024, 4, 14, 16, 16, 37, 922709), datetime_complete=datetime.datetime(2024, 4, 14, 16, 16, 38, 67103), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.5957320931913246


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709975125414
Fold 2 IBS: 0.23203988383665525
Fold 3 IBS: 0.22898186801315143
Fold 4 IBS: 0.24197477015984858
Fold 5 IBS: 0.22939559219095101
[I 2024-04-14 16:16:38,267] Trial 0 finished with value: 0.23592784279037207 and parameters: {}. Best is trial 0 with value: 0.23592784279037207.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592784279037207], datetime_start=datetime.datetime(2024, 4, 14, 16, 16, 38, 100377), datetime_complete=datetime.datetime(2024, 4, 14, 16, 16, 38, 267307), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592784279037207


In [35]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [36]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.596
train_ibs:  0.236


#### Test

In [37]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [38]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.51


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [39]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [40]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 16:16:41,714] A new study created in memory with name: no-name-87a61a6e-865e-4449-b5d0-909350624c9d


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.40239043824701193
Fold 2 C-index: 0.6046511627906976
Fold 3 C-index: 0.37446808510638296
Fold 4 C-index: 0.38022813688212925


[I 2024-04-14 16:16:43,123] A new study created in memory with name: no-name-71d35dfb-01e2-4338-a6ad-742361194801


Fold 5 C-index: 0.5793991416309013
[I 2024-04-14 16:16:43,117] Trial 0 finished with value: 0.4682273929314246 and parameters: {}. Best is trial 0 with value: 0.4682273929314246.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.4682273929314246], datetime_start=datetime.datetime(2024, 4, 14, 16, 16, 41, 747800), datetime_complete=datetime.datetime(2024, 4, 14, 16, 16, 43, 117545), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.4682273929314246


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.5098830107005321
Fold 2 IBS: 0.3372965635980787
Fold 3 IBS: 0.4733678688584834
Fold 4 IBS: 0.5026474555138712
Fold 5 IBS: 0.3423885899507013
[I 2024-04-14 16:16:44,733] Trial 0 finished with value: 0.43311669772433337 and parameters: {}. Best is trial 0 with value: 0.43311669772433337.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.43311669772433337], datetime_start=datetime.datetime(2024, 4, 14, 16, 16, 43, 156238), datetime_complete=datetime.datetime(2024, 4, 14, 16, 16, 44, 733045), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.43311669772433337


In [41]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [42]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.468
train_ibs:  0.433


#### Test

In [43]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [44]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.522


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.423


In [45]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [46]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 16:16:45,448] A new study created in memory with name: no-name-33e8e44c-b065-4209-a3fd-8c4b182aa266


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.41434262948207173
Fold 2 C-index: 0.5813953488372093
Fold 3 C-index: 0.40425531914893614
Fold 4 C-index: 0.43346007604562736
Fold 5 C-index: 0.5536480686695279
[I 2024-04-14 16:16:46,284] Trial 0 finished with value: 0.4774202884366745 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.4774202884366745.
Fold 1 C-index: 0.42231075697211157
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.688212927756654
Fold 5 C-index: 0.6738197424892703
[I 2024-04-14 16:16:46,563] Trial 1 finished with value: 0.6110695760918012 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.6110695760918012.
Fold 1 C-index: 0.5856573705179283
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.688212927756654
Fold 5 C-index: 0.6738197424892703
[I 2024-04-14 16:16:46,720] Trial 2 finished with value: 0.6437388988009645 and parameters: {'l1_ratio': 0.22692876841884668

Fold 1 C-index: 0.41434262948207173
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6844106463878327
Fold 5 C-index: 0.5793991416309013
[I 2024-04-14 16:16:54,603] Trial 24 finished with value: 0.5898313741483551 and parameters: {'l1_ratio': 0.36894274155306017}. Best is trial 2 with value: 0.6437388988009645.
Fold 1 C-index: 0.42231075697211157
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.688212927756654
Fold 5 C-index: 0.6738197424892703
[I 2024-04-14 16:16:54,879] Trial 25 finished with value: 0.6110695760918012 and parameters: {'l1_ratio': 0.28250483325041786}. Best is trial 2 with value: 0.6437388988009645.
Fold 1 C-index: 0.41832669322709165
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.5057034220532319
Fold 5 C-index: 0.575107296137339
[I 2024-04-14 16:16:55,374] Trial 26 finished with value: 0.5540283729317265 and parameters: {'l1_ratio': 0.43696099753129

Fold 1 C-index: 0.41434262948207173
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6844106463878327
Fold 5 C-index: 0.5793991416309013
[I 2024-04-14 16:17:03,763] Trial 48 finished with value: 0.5898313741483551 and parameters: {'l1_ratio': 0.3557590214625724}. Best is trial 2 with value: 0.6437388988009645.
Fold 1 C-index: 0.5816733067729084
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6920152091254753
Fold 5 C-index: 0.648068669527897
[I 2024-04-14 16:17:03,959] Trial 49 finished with value: 0.6385523277334502 and parameters: {'l1_ratio': 0.04567662268230008}. Best is trial 2 with value: 0.6437388988009645.
Fold 1 C-index: 0.41832669322709165
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6844106463878327
Fold 5 C-index: 0.5708154506437768
[I 2024-04-14 16:17:04,343] Trial 50 finished with value: 0.5889114486999342 and parameters: {'l1_ratio': 0.416539858580717

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.49645356464358287
Fold 2 IBS: 0.30808905655404734
Fold 3 IBS: 0.36277782336061554
Fold 4 IBS: 0.4089481533297419
Fold 5 IBS: 0.3412191759164054
[I 2024-04-14 16:17:18,606] Trial 0 finished with value: 0.3834975547608786 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.3834975547608786.
Fold 1 IBS: 0.4508594624119911
Fold 2 IBS: 0.2298052971832474
Fold 3 IBS: 0.22882247544561968
Fold 4 IBS: 0.23850171003694137
Fold 5 IBS: 0.2268433170424036
[I 2024-04-14 16:17:18,930] Trial 1 finished with value: 0.2749664524240406 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.2749664524240406.
Fold 1 IBS: 0.24564380086531118
Fold 2 IBS: 0.2303205034122087
Fold 3 IBS: 0.2288540127794953
Fold 4 IBS: 0.23918299331599008
Fold 5 IBS: 0.22739419016239
[I 2024-04-14 16:17:19,201] Trial 2 finished with value: 0.23427910010707906 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.23427910010707906.
Fold

Fold 1 IBS: 0.466398795642302
Fold 2 IBS: 0.22882772639684487
Fold 3 IBS: 0.22877290473956685
Fold 4 IBS: 0.2373532638825162
Fold 5 IBS: 0.29681562006401563
[I 2024-04-14 16:17:29,727] Trial 25 finished with value: 0.29163366214504916 and parameters: {'l1_ratio': 0.38971525832834536}. Best is trial 24 with value: 0.23401139302043905.
Fold 1 IBS: 0.45186227275074664
Fold 2 IBS: 0.22976130645640724
Fold 3 IBS: 0.2288199466463936
Fold 4 IBS: 0.2384461837034837
Fold 5 IBS: 0.2267970258922456
[I 2024-04-14 16:17:30,053] Trial 26 finished with value: 0.2751373470898554 and parameters: {'l1_ratio': 0.2911123931485357}. Best is trial 24 with value: 0.23401139302043905.
Fold 1 IBS: 0.47624166338835305
Fold 2 IBS: 0.22784726392344626
Fold 3 IBS: 0.22873875445233233
Fold 4 IBS: 0.37634204248349573
Fold 5 IBS: 0.3109773857303955
[I 2024-04-14 16:17:30,581] Trial 27 finished with value: 0.3240294219956046 and parameters: {'l1_ratio': 0.48333206147305935}. Best is trial 24 with value: 0.234011393020

Fold 1 IBS: 0.24637482995821008
Fold 2 IBS: 0.2311347354827766
Fold 3 IBS: 0.22891109863503797
Fold 4 IBS: 0.24040460607923667
Fold 5 IBS: 0.22831374870955315
[I 2024-04-14 16:17:41,053] Trial 50 finished with value: 0.2350278037729629 and parameters: {'l1_ratio': 0.12455647091111496}. Best is trial 48 with value: 0.2339125815801168.
Fold 1 IBS: 0.45336290866765294
Fold 2 IBS: 0.2296841697272185
Fold 3 IBS: 0.22881557701400743
Fold 4 IBS: 0.2383497652205923
Fold 5 IBS: 0.2267161104254823
[I 2024-04-14 16:17:41,469] Trial 51 finished with value: 0.2753857062109907 and parameters: {'l1_ratio': 0.29964893239692797}. Best is trial 48 with value: 0.2339125815801168.
Fold 1 IBS: 0.24594423385734104
Fold 2 IBS: 0.23066584735776324
Fold 3 IBS: 0.2288770356389336
Fold 4 IBS: 0.23967447114034615
Fold 5 IBS: 0.22777375198236005
[I 2024-04-14 16:17:41,707] Trial 52 finished with value: 0.23458706799534879 and parameters: {'l1_ratio': 0.18514345548822425}. Best is trial 48 with value: 0.23391258158

Fold 2 IBS: 0.23046625016103817
Fold 3 IBS: 0.22886355164113878
Fold 4 IBS: 0.23938678822922885
Fold 5 IBS: 0.22755326987902014
[I 2024-04-14 16:17:53,599] Trial 75 finished with value: 0.23440786844886544 and parameters: {'l1_ratio': 0.20950302481432298}. Best is trial 48 with value: 0.2339125815801168.
Fold 1 IBS: 0.4580818271335593
Fold 2 IBS: 0.2294254508419846
Fold 3 IBS: 0.22880153462379343
Fold 4 IBS: 0.2380348649875556
Fold 5 IBS: 0.22644694102617802
[I 2024-04-14 16:17:54,184] Trial 76 finished with value: 0.27615812372261417 and parameters: {'l1_ratio': 0.32774984782502226}. Best is trial 48 with value: 0.2339125815801168.
Fold 1 IBS: 0.474858623807836
Fold 2 IBS: 0.22801368333023353
Fold 3 IBS: 0.22874333579261105
Fold 4 IBS: 0.374315539258652
Fold 5 IBS: 0.3086153091428531
[I 2024-04-14 16:17:54,997] Trial 77 finished with value: 0.32290929826643716 and parameters: {'l1_ratio': 0.4680890068967426}. Best is trial 48 with value: 0.2339125815801168.
Fold 1 IBS: 0.4997659336752

In [47]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [48]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.644
train_ibs:  0.234


#### Test

In [49]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [50]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.22692876841884668)

test_cindex : 0.532


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.2761236619004883)

test_ibs:  0.228


In [51]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [52]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 16:18:05,164] A new study created in memory with name: no-name-dd0da468-e944-4bd8-bc91-7f0e07c2f0aa


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.7558139534883721
Fold 3 C-index: 0.6425531914893617
Fold 4 C-index: 0.7186311787072244
Fold 5 C-index: 0.6523605150214592
[I 2024-04-14 16:18:23,752] Trial 0 finished with value: 0.676580931087897 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.676580931087897.
Fold 1 C-index: 0.5537848605577689
Fold 2 C-index: 0.6744186046511628
Fold 3 C-index: 0.6893617021276596
Fold 4 C-index: 0.6273764258555133
Fold 5 C-index: 0.5278969957081545
[I 2024-04-14 16:18:27,243] Trial 1 finished with value: 0.6145677177800518 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'm

Fold 5 C-index: 0.6824034334763949
[I 2024-04-14 16:19:24,300] Trial 15 finished with value: 0.7109217554965979 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 7, 'min_samples_leaf': 16, 'max_depth': 6, 'n_estimators': 96, 'oob_score': True, 'max_samples': 0.8147833133306369, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.2047601680710191, 'warm_start': True}. Best is trial 15 with value: 0.7109217554965979.
Fold 1 C-index: 0.3804780876494024
Fold 2 C-index: 0.810077519379845
Fold 3 C-index: 0.7936170212765957
Fold 4 C-index: 0.5836501901140685
Fold 5 C-index: 0.6351931330472103
[I 2024-04-14 16:19:24,466] Trial 16 finished with value: 0.6406031902934244 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 2, 'min_samples_leaf': 18, 'max_depth': 6, 'n_estimators': 5, 'oob_score': True, 'max_samples': 0.8484814588885312, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.3777228897992014, 'warm_start': True}. Best is trial 15 with value: 0.7109217554965979.


Fold 1 C-index: 0.549800796812749
Fold 2 C-index: 0.8255813953488372
Fold 3 C-index: 0.8680851063829788
Fold 4 C-index: 0.8288973384030418
Fold 5 C-index: 0.8111587982832618
[I 2024-04-14 16:19:39,021] Trial 30 finished with value: 0.7767046870461737 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 9, 'min_samples_leaf': 10, 'max_depth': 12, 'n_estimators': 122, 'oob_score': True, 'max_samples': 0.6865809791097957, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.03407840564085754, 'warm_start': True}. Best is trial 30 with value: 0.7767046870461737.
Fold 1 C-index: 0.549800796812749
Fold 2 C-index: 0.8294573643410853
Fold 3 C-index: 0.8723404255319149
Fold 4 C-index: 0.8288973384030418
Fold 5 C-index: 0.8111587982832618
[I 2024-04-14 16:19:40,121] Trial 31 finished with value: 0.7783309446744106 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 9, 'min_samples_leaf': 10, 'max_depth': 12, 'n_estimators': 128, 'oob_score': True, 'max_samples': 0.686417287257043

...

Fold 1 IBS: 0.223671117899065
Fold 2 IBS: 0.18242513973526725
Fold 3 IBS: 0.2389733570256732
Fold 4 IBS: 0.21132681423071092
Fold 5 IBS: 0.2129200403370242
[I 2024-04-14 16:42:24,833] Trial 32 finished with value: 0.2138632938455481 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 11, 'min_samples_leaf': 8, 'max_depth': 14, 'n_estimators': 294, 'oob_score': True, 'max_samples': 0.9673759674409311, 'max_features': None, 'min_weight_fraction_leaf': 0.27417727195073677}. Best is trial 32 with value: 0.2138632938455481.
Fold 1 IBS: 0.2573672754798652
Fold 2 IBS: 0.1902152831375234
Fold 3 IBS: 0.2272585883751577
Fold 4 IBS: 0.22496428138050273
Fold 5 IBS: 0.2133258411787376
[I 2024-04-14 16:43:12,356] Trial 33 finished with value: 0.2226262539103573 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 13, 'min_samples_leaf': 13, 'max_depth': 13, 'n_estimators': 304, 'oob_score': True, 'max_samples': 0.9964442664551478, 'max_features': None, 'min_weight_fraction_leaf': 0.185

Fold 1 IBS: 0.23136140224576346
Fold 2 IBS: 0.1874927380948624
Fold 3 IBS: 0.2417750554443432
Fold 4 IBS: 0.2265069844402473
Fold 5 IBS: 0.2218959262967509
[I 2024-04-14 16:47:55,978] Trial 48 finished with value: 0.22180642130439346 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 15, 'n_estimators': 354, 'oob_score': True, 'max_samples': 0.9047670799898727, 'max_features': None, 'min_weight_fraction_leaf': 0.31310643491678164}. Best is trial 32 with value: 0.2138632938455481.
Fold 1 IBS: 0.23542420632037062
Fold 2 IBS: 0.21022618815946087
Fold 3 IBS: 0.24786241576097612
Fold 4 IBS: 0.24693634633915879
Fold 5 IBS: 0.2275366959349153
[I 2024-04-14 16:48:04,150] Trial 49 finished with value: 0.23359717050297632 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 20, 'max_depth': 13, 'n_estimators': 126, 'oob_score': True, 'max_samples': 0.7434493023402162, 'max_features': None, 'min_weight_fraction_leaf': 0

Fold 1 IBS: 0.23011155909129694
Fold 2 IBS: 0.18057364325153813
Fold 3 IBS: 0.23750151803668482
Fold 4 IBS: 0.21433948253554103
Fold 5 IBS: 0.21274097244510462
[I 2024-04-14 16:55:34,082] Trial 64 finished with value: 0.2150534350720331 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 8, 'min_samples_leaf': 4, 'max_depth': 17, 'n_estimators': 249, 'oob_score': True, 'max_samples': 0.9368121127684691, 'max_features': None, 'min_weight_fraction_leaf': 0.24195064098385735}. Best is trial 32 with value: 0.2138632938455481.
Fold 1 IBS: 0.22994947133324334
Fold 2 IBS: 0.1928070595691341
Fold 3 IBS: 0.24608508712178995
Fold 4 IBS: 0.22580322695561086
Fold 5 IBS: 0.2205885428532394
[I 2024-04-14 16:55:58,960] Trial 65 finished with value: 0.2230466775666035 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 9, 'min_samples_leaf': 3, 'max_depth': 13, 'n_estimators': 318, 'oob_score': True, 'max_samples': 0.8670083844909762, 'max_features': None, 'min_weight_fraction_leaf': 0.

Fold 1 IBS: 0.23374541142463764
Fold 2 IBS: 0.19941476694876598
Fold 3 IBS: 0.24370640781635464
Fold 4 IBS: 0.23069544004770481
Fold 5 IBS: 0.2221664668271361
[I 2024-04-14 17:02:03,825] Trial 80 finished with value: 0.22594569861291985 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 13, 'min_samples_leaf': 4, 'max_depth': 17, 'n_estimators': 303, 'oob_score': True, 'max_samples': 0.6585925147157417, 'max_features': None, 'min_weight_fraction_leaf': 0.2649590538103387}. Best is trial 32 with value: 0.2138632938455481.
Fold 1 IBS: 0.22301744452662423
Fold 2 IBS: 0.18261690178298304
Fold 3 IBS: 0.24046916847175354
Fold 4 IBS: 0.21144194863813523
Fold 5 IBS: 0.21301702675170428
[I 2024-04-14 17:02:36,282] Trial 81 finished with value: 0.21411249803424007 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 10, 'min_samples_leaf': 2, 'max_depth': 16, 'n_estimators': 273, 'oob_score': True, 'max_samples': 0.9753152468491701, 'max_features': None, 'min_weight_fraction_leaf'

Fold 1 IBS: 0.23083721945133412
Fold 2 IBS: 0.18442690744496573
Fold 3 IBS: 0.2372912055911599
Fold 4 IBS: 0.22822622495739184
Fold 5 IBS: 0.22366315254862057
[I 2024-04-14 17:08:57,560] Trial 96 finished with value: 0.22088894199869444 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 10, 'min_samples_leaf': 3, 'max_depth': 14, 'n_estimators': 231, 'oob_score': True, 'max_samples': 0.9522685325324347, 'max_features': None, 'min_weight_fraction_leaf': 0.32080383117786954}. Best is trial 91 with value: 0.21308616320039436.
Fold 1 IBS: 0.2265678317463007
Fold 2 IBS: 0.18314595561796107
Fold 3 IBS: 0.2376073017283759
Fold 4 IBS: 0.21337826003109
Fold 5 IBS: 0.21787901230329237
[I 2024-04-14 17:09:09,556] Trial 97 finished with value: 0.21571567228540403 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 11, 'min_samples_leaf': 10, 'max_depth': 17, 'n_estimators': 174, 'oob_score': True, 'max_samples': 0.9189184529013809, 'max_features': None, 'min_weight_fraction_leaf': 

In [53]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [54]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.826
train_ibs:  0.213


#### Test

In [55]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

In [56]:
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])
 
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=13, max_features=None, max_leaf_nodes=10,
                     max_samples=0.750348183174299, min_samples_leaf=2,
                     min_samples_split=16,
                     min_weight_fraction_leaf=0.09298348876429541,
                     n_estimators=267, random_state=123, warm_start=True)

test_cindex:  0.516


RandomSurvivalForest(max_depth=5, max_features=None, max_leaf_nodes=13,
                     max_samples=0.9824332970054127, min_samples_split=10,
                     min_weight_fraction_leaf=0.296758543375246,
                     n_estimators=236, oob_score=True, random_state=123)

test_ibs:  0.258


In [57]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [58]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [59]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 17:09:58,003] A new study created in memory with name: no-name-9ad15175-278c-4899-ae2b-306527a5cceb


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.4581673306772908
Fold 2 C-index: 0.7829457364341085
Fold 3 C-index: 0.8382978723404255
Fold 4 C-index: 0.688212927756654
Fold 5 C-index: 0.6909871244635193
[I 2024-04-14 17:09:59,439] Trial 0 finished with value: 0.6917221983343996 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.6917221983343996.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 17:10:04,073] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}

Fold 1 C-index: 0.47808764940239046
Fold 2 C-index: 0.7596899224806202
Fold 3 C-index: 0.825531914893617
Fold 4 C-index: 0.6920152091254753
Fold 5 C-index: 0.6394849785407726
[I 2024-04-14 17:10:26,836] Trial 16 finished with value: 0.678961934888575 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 10, 'min_samples_leaf': 13, 'max_depth': 10, 'n_estimators': 95, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.884183184043622, 'min_weight_fraction_leaf': 0.10581506507448754}. Best is trial 8 with value: 0.7123582035090831.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 17:10:27,297] Trial 17 finished with value: 0.5 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 15, 'min_samples_leaf': 9, 'max_depth': 4, 'n_estimators': 162, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.37217752523657055, 'min_weight_fraction_leaf': 0.1994767874105

Fold 1 C-index: 0.47410358565737054
Fold 2 C-index: 0.7713178294573644
Fold 3 C-index: 0.8297872340425532
Fold 4 C-index: 0.6730038022813688
Fold 5 C-index: 0.6523605150214592
[I 2024-04-14 17:10:35,625] Trial 31 finished with value: 0.6801145932920232 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 4, 'min_samples_leaf': 8, 'max_depth': 13, 'n_estimators': 362, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.5225620293165905, 'min_weight_fraction_leaf': 0.0017285001202470562}. Best is trial 20 with value: 0.7286697682430041.
Fold 1 C-index: 0.49800796812749004
Fold 2 C-index: 0.7596899224806202
Fold 3 C-index: 0.825531914893617
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6437768240343348
[I 2024-04-14 17:10:36,550] Trial 32 finished with value: 0.6731579798996079 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 3, 'min_samples_leaf': 7, 'max_depth': 13, 'n_estimators': 372, 'oob_score': False, 'warm_start': True, 'max_feat

...

Fold 1 IBS: 0.23497354688702718
Fold 2 IBS: 0.1976009711176378
Fold 3 IBS: 0.20496554854896726
Fold 4 IBS: 0.22195679558988315
Fold 5 IBS: 0.22437156955219212
[I 2024-04-14 17:18:34,733] Trial 30 finished with value: 0.21677368633914149 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 15, 'min_samples_leaf': 13, 'max_depth': 12, 'n_estimators': 351, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.8683525444021144, 'min_weight_fraction_leaf': 0.10877158322577073}. Best is trial 30 with value: 0.21677368633914149.
Fold 1 IBS: 0.23741283004501632
Fold 2 IBS: 0.19660814788429976
Fold 3 IBS: 0.208320965032862
Fold 4 IBS: 0.22424875267364808
Fold 5 IBS: 0.22365262373167072
[I 2024-04-14 17:18:41,703] Trial 31 finished with value: 0.2180486638734994 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 15, 'min_samples_leaf': 13, 'max_depth': 12, 'n_estimators': 347, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 

Fold 1 IBS: 0.23322825812700776
Fold 2 IBS: 0.19980996308476648
Fold 3 IBS: 0.20366753393387363
Fold 4 IBS: 0.23327551134614005
Fold 5 IBS: 0.2261815525696507
[I 2024-04-14 17:20:14,305] Trial 45 finished with value: 0.2192325638122877 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 16, 'min_samples_leaf': 14, 'max_depth': 11, 'n_estimators': 369, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.7548177510477919, 'min_weight_fraction_leaf': 0.13669728117144694}. Best is trial 36 with value: 0.2167639286316539.
Fold 1 IBS: 0.2465213089401209
Fold 2 IBS: 0.23046298845066793
Fold 3 IBS: 0.22971391168769142
Fold 4 IBS: 0.24253981917172043
Fold 5 IBS: 0.23157320473775703
[I 2024-04-14 17:20:18,320] Trial 46 finished with value: 0.23616224659759152 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 14, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 314, 'oob_score': True, 'warm_start': True, 'max_features': 1, 'max_samples': 0.8165

Fold 1 IBS: 0.24675120399960646
Fold 2 IBS: 0.2238312246331017
Fold 3 IBS: 0.22373605345231745
Fold 4 IBS: 0.24763583159134048
Fold 5 IBS: 0.2361001387256771
[I 2024-04-14 17:21:29,640] Trial 60 finished with value: 0.23561089048040867 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 20, 'min_samples_leaf': 5, 'max_depth': 5, 'n_estimators': 259, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.5876523474241502, 'min_weight_fraction_leaf': 0.18336534674120722}. Best is trial 53 with value: 0.21672142772101383.
Fold 1 IBS: 0.24544383395694797
Fold 2 IBS: 0.19103401818321838
Fold 3 IBS: 0.1977979215082493
Fold 4 IBS: 0.2360015957855052
Fold 5 IBS: 0.22738640071735247
[I 2024-04-14 17:21:39,810] Trial 61 finished with value: 0.21953275403025466 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 16, 'min_samples_leaf': 3, 'max_depth': 6, 'n_estimators': 363, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.98

Fold 1 IBS: 0.23878866729576506
Fold 2 IBS: 0.19862339947767513
Fold 3 IBS: 0.20582989519828723
Fold 4 IBS: 0.2244448023517984
Fold 5 IBS: 0.2249714920857591
[I 2024-04-14 17:23:12,672] Trial 75 finished with value: 0.218531651281857 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 13, 'min_samples_leaf': 4, 'max_depth': 14, 'n_estimators': 284, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.8355109212222066, 'min_weight_fraction_leaf': 0.1496381039654338}. Best is trial 53 with value: 0.21672142772101383.
Fold 1 IBS: 0.24638426866097177
Fold 2 IBS: 0.22911008901873028
Fold 3 IBS: 0.22825388044344033
Fold 4 IBS: 0.2429147576831134
Fold 5 IBS: 0.2310867513895085
[I 2024-04-14 17:23:16,940] Trial 76 finished with value: 0.23554994943915286 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 17, 'min_samples_leaf': 13, 'max_depth': 10, 'n_estimators': 402, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.9

Fold 1 IBS: 0.23711736196146355
Fold 2 IBS: 0.20354697260954158
Fold 3 IBS: 0.20782994592008056
Fold 4 IBS: 0.235467444322929
Fold 5 IBS: 0.23168859841356962
[I 2024-04-14 17:24:41,233] Trial 90 finished with value: 0.22313006464551685 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 17, 'min_samples_leaf': 13, 'max_depth': 14, 'n_estimators': 239, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.6267661705142946, 'min_weight_fraction_leaf': 0.049594043056366624}. Best is trial 53 with value: 0.21672142772101383.
Fold 1 IBS: 0.24102633692552217
Fold 2 IBS: 0.19520843787050352
Fold 3 IBS: 0.20356979583511886
Fold 4 IBS: 0.2204498673203032
Fold 5 IBS: 0.22696028171118432
[I 2024-04-14 17:24:46,980] Trial 91 finished with value: 0.21744294393252642 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 20, 'min_samples_leaf': 10, 'max_depth': 13, 'n_estimators': 304, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.

In [60]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [61]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.859
train_ibs:  0.217


#### Test

In [62]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [63]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=13, max_features=None, max_leaf_nodes=19,
                   max_samples=0.9533554910566981,
                   min_weight_fraction_leaf=0.01276054813110837,
                   n_estimators=487, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.506


ExtraSurvivalTrees(max_depth=6, max_features=None, max_leaf_nodes=18,
                   max_samples=0.9463423717095281, min_samples_leaf=4,
                   min_samples_split=14,
                   min_weight_fraction_leaf=0.17857938204264587,
                   n_estimators=273, oob_score=True, random_state=123)

IBS: 0.245


In [64]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [65]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 17:25:32,661] A new study created in memory with name: no-name-fd8763ef-1ea4-4848-a9eb-338f6a4be5a7


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 17:26:21,431] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 17:26:48,155] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 17:36:02,905] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.6859109422281264.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 17:37:09,230] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 17:50:24,369] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8502968404151126, 'learning_rate': 0.024443951259730985, 'dropout_rate': 0.2675273969344081, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 2.0753749717266823, 'min_weight_fraction_leaf': 0.3503497788125578, 'max_features': 'auto', 'min_impurity_decrease': 7.237153572123947e-07, 'validation_fraction': 0.41222573804914475, 'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 9 with value: 0.6859109422281264.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 17:50:52,648] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7670085127536703, 'learning_rate': 0.011828778593594373, 'dropout_rate': 0.4300954216773497, 'n_estimators': 330, 'criterion': 'friedman_mse', 'ccp_alpha':

...

Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 20:27:36,385] Trial 44 finished with value: 0.2359278435123307 and parameters: {'subsample': 0.9992871622667048, 'learning_rate': 0.012272887594565313, 'dropout_rate': 0.11965546339368549, 'n_estimators': 451, 'criterion': 'squared_error', 'ccp_alpha': 0.5504303626596221, 'min_weight_fraction_leaf': 0.10316812300893248, 'max_features': 'auto', 'min_impurity_decrease': 2.2780695231073978e-06, 'validation_fraction': 0.8946516967835402, 'min_samples_split': 17, 'max_leaf_nodes': 14, 'min_samples_leaf': 10, 'max_depth': 2}. Best is trial 41 with value: 0.23454270328649618.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 20:28:07,711] Trial 45 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8888336111822305, 'learning_rate': 0.022080051602678542, 'dropout_rate': 0.2121382569959027, 'n_estimators': 2

Fold 5 IBS: 0.22939559304809248
[I 2024-04-14 20:42:12,054] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.809323041898869, 'learning_rate': 0.0040826970179266685, 'dropout_rate': 0.36065535241543395, 'n_estimators': 435, 'criterion': 'squared_error', 'ccp_alpha': 1.5126136571072866, 'min_weight_fraction_leaf': 0.2002957617254777, 'max_features': 'auto', 'min_impurity_decrease': 2.0868015436948727e-05, 'validation_fraction': 0.8814649292460881, 'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'max_depth': 2}. Best is trial 41 with value: 0.23454270328649618.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-14 20:43:29,102] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.2818850250060197, 'learning_rate': 0.09850922090204048, 'dropout_rate': 0.23297170241739207, 'n_estimators':

Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 20:58:48,872] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8489175059797514, 'learning_rate': 0.008448226967076113, 'dropout_rate': 0.12725750793823018, 'n_estimators': 424, 'criterion': 'squared_error', 'ccp_alpha': 0.893877768639663, 'min_weight_fraction_leaf': 0.2702972473075093, 'max_features': 'auto', 'min_impurity_decrease': 7.586810829424823e-05, 'validation_fraction': 0.8971088342813041, 'min_samples_split': 17, 'max_leaf_nodes': 17, 'min_samples_leaf': 8, 'max_depth': 4}. Best is trial 62 with value: 0.2343598469061142.
Fold 1 IBS: 0.24724710044658996
Fold 2 IBS: 0.23203988453792296
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 21:00:23,632] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9594402941587806, 'learning_rate': 0.0039206688180527, 'dropout_rate': 0.25338876644885766, 'n_estimators': 444, '

Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 21:14:49,993] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6708593402641205, 'learning_rate': 0.02742354559286264, 'dropout_rate': 0.12642193192647372, 'n_estimators': 329, 'criterion': 'squared_error', 'ccp_alpha': 0.2456295975030462, 'min_weight_fraction_leaf': 0.3629864105918121, 'max_features': None, 'min_impurity_decrease': 0.00015140034318578482, 'validation_fraction': 0.6173345703129903, 'min_samples_split': 10, 'max_leaf_nodes': 8, 'min_samples_leaf': 11, 'max_depth': 5}. Best is trial 70 with value: 0.2341005897890583.
Fold 1 IBS: 0.2422361212718552
Fold 2 IBS: 0.22571122387519038
Fold 3 IBS: 0.22923653706299701
Fold 4 IBS: 0.2398268927530749
Fold 5 IBS: 0.2262157333144423
[I 2024-04-14 21:16:04,958] Trial 79 finished with value: 0.23264530165551198 and parameters: {'sub

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 21:24:42,226] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6555693108457097, 'learning_rate': 0.08348043724232247, 'dropout_rate': 0.1018982518393906, 'n_estimators': 126, 'criterion': 'squared_error', 'ccp_alpha': 5.474477626804302, 'min_weight_fraction_leaf': 0.3922659564335438, 'max_features': None, 'min_impurity_decrease': 1.84353439530822e-05, 'validation_fraction': 0.4550485507485021, 'min_samples_split': 17, 'max_leaf_nodes': 3, 'min_samples_leaf': 6, 'max_depth': 9}. Best is trial 81 with value: 0.23126689637824502.
Fold 1 IBS: 0.24724710044658996
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-14 21:25:00,614] Trial 90 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6142534760575815, 'lear

In [66]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [67]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.686
train_ibs:  0.231


#### Test

In [68]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [69]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.07426378544613033,
                                 criterion='squared_error',
                                 dropout_rate=0.28125955124323376,
                                 learning_rate=0.013102111413618,
                                 max_features='auto', max_leaf_nodes=16,
                                 min_impurity_decrease=1.4994028685178666e-07,
                                 min_samples_leaf=10,
                                 min_weight_fraction_leaf=0.27579636299120275,
                                 n_estimators=406, random_state=123,
                                 subsample=0.6059965408578151,
                                 validation_fraction=0.6359003593513561)

C-index score: 0.518


GradientBoostingSurvivalAnalysis(ccp_alpha=0.03787089343697578,
                                 criterion='squared_error',
                                 dropout_rate=0.18479264516541616,
                                 learning_rate=0.09426091382038485, max_depth=2,
                                 max_leaf_nodes=6,
                                 min_impurity_decrease=7.4293259533530295e-06,
                                 min_samples_leaf=7, min_samples_split=18,
                                 min_weight_fraction_leaf=0.4036747584607865,
                                 n_estimators=390, random_state=123,
                                 subsample=0.579213280645811,
                                 validation_fraction=0.5325834925366492)

IBS: 0.23


In [70]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [71]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [72]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 21:32:34,002] A new study created in memory with name: no-name-43934bc3-ce57-42b5-9517-7f78e14e2d11


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5298804780876494
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.5446808510638298
Fold 4 C-index: 0.45627376425855515
Fold 5 C-index: 0.5236051502145923
[I 2024-04-14 21:32:40,828] Trial 0 finished with value: 0.5527485138412043 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.5527485138412043.
Fold 1 C-index: 0.5219123505976095
Fold 2 C-index: 0.7131782945736435
Fold 3 C-index: 0.548936170212766
Fold 4 C-index: 0.45627376425855515
Fold 5 C-index: 0.51931330472103
[I 2024-04-14 21:33:17,050] Trial 1 finished with value: 0.5519227768727208 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.5527485138412043.
Fold 1 C-index: 0.5139442231075697
Fold 2 C-index: 0.7170542635658915
Fold 3 C-index: 0.6170212765957447
Fold 4

Fold 1 C-index: 0.5338645418326693
Fold 2 C-index: 0.7170542635658915
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.45627376425855515
Fold 5 C-index: 0.5236051502145923
[I 2024-04-14 21:36:58,035] Trial 19 finished with value: 0.5584999695062565 and parameters: {'subsample': 0.8350722272141606, 'dropout_rate': 0.9949834936920623, 'n_estimators': 134, 'learning_rate': 0.036212385990032875}. Best is trial 10 with value: 0.5648744714982884.
Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.7170542635658915
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.45627376425855515
Fold 5 C-index: 0.5236051502145923
[I 2024-04-14 21:37:02,497] Trial 20 finished with value: 0.5656712842472924 and parameters: {'subsample': 0.9139336698304192, 'dropout_rate': 0.6485024664008028, 'n_estimators': 61, 'learning_rate': 0.017331004877487094}. Best is trial 20 with value: 0.5656712842472924.
Fold 1 C-index: 0.5657370517928287
Fold 2 C-index: 0.7170542635658915
Fold 3 C-index: 0.561702127659574

Fold 3 C-index: 0.3872340425531915
Fold 4 C-index: 0.45627376425855515
Fold 5 C-index: 0.6008583690987125
[I 2024-04-14 21:39:18,814] Trial 38 finished with value: 0.5185776732437984 and parameters: {'subsample': 0.3866096773460963, 'dropout_rate': 0.3010386389757013, 'n_estimators': 1, 'learning_rate': 0.06351044805702724}. Best is trial 20 with value: 0.5656712842472924.
Fold 1 C-index: 0.5298804780876494
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.45627376425855515
Fold 5 C-index: 0.5236051502145923
[I 2024-04-14 21:39:31,103] Trial 39 finished with value: 0.5576272867259149 and parameters: {'subsample': 0.8328204942141013, 'dropout_rate': 0.8890898259545601, 'n_estimators': 168, 'learning_rate': 0.0969464828585424}. Best is trial 20 with value: 0.5656712842472924.
Fold 1 C-index: 0.5537848605577689
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.45627376425855515
Fold 5 C-index: 0.5236051502145923
[

...

Fold 2 IBS: 0.217623028684295
Fold 3 IBS: 0.22928372477732015
Fold 4 IBS: 0.30100053255938797
Fold 5 IBS: 0.2325559394582789
[I 2024-04-14 21:52:54,667] Trial 19 finished with value: 0.24654446514731837 and parameters: {'subsample': 0.650899739648275, 'dropout_rate': 0.9954484919483726, 'n_estimators': 400, 'learning_rate': 0.011730584939170362}. Best is trial 13 with value: 0.23593001841880062.
Fold 1 IBS: 0.244884130972059
Fold 2 IBS: 0.22525640865538507
Fold 3 IBS: 0.2287419458584932
Fold 4 IBS: 0.26450177998590696
Fold 5 IBS: 0.2290688156699557
[I 2024-04-14 21:52:59,279] Trial 20 finished with value: 0.23849061622836 and parameters: {'subsample': 0.9010126869682862, 'dropout_rate': 0.6173032374045795, 'n_estimators': 67, 'learning_rate': 0.0257290746252228}. Best is trial 13 with value: 0.23593001841880062.
Fold 1 IBS: 0.2472013998837189
Fold 2 IBS: 0.2319637496979315
Fold 3 IBS: 0.2289750350168313
Fold 4 IBS: 0.24211711207007558
Fold 5 IBS: 0.22938745518196135
[I 2024-04-14 21:53

Fold 3 IBS: 0.22982813627745377
Fold 4 IBS: 0.2982564284360105
Fold 5 IBS: 0.23151273581044918
[I 2024-04-14 21:54:49,911] Trial 38 finished with value: 0.2456892180069786 and parameters: {'subsample': 0.7343497854758639, 'dropout_rate': 0.13272164755980653, 'n_estimators': 79, 'learning_rate': 0.060691705645777076}. Best is trial 22 with value: 0.2359281786560819.
Fold 1 IBS: 0.25017095026188413
Fold 2 IBS: 0.21870305039650859
Fold 3 IBS: 0.23036639060624442
Fold 4 IBS: 0.29742908883540964
Fold 5 IBS: 0.23185851227205606
[I 2024-04-14 21:55:01,337] Trial 39 finished with value: 0.24570559847442058 and parameters: {'subsample': 0.8328204942141013, 'dropout_rate': 0.5841264351271549, 'n_estimators': 159, 'learning_rate': 0.02762747332143923}. Best is trial 22 with value: 0.2359281786560819.
Fold 1 IBS: 0.2449422956152037
Fold 2 IBS: 0.22636241594032222
Fold 3 IBS: 0.22872398892904824
Fold 4 IBS: 0.259545522434371
Fold 5 IBS: 0.2290526113095854
[I 2024-04-14 21:55:03,481] Trial 40 finish

Fold 4 IBS: 0.24395238716462248
Fold 5 IBS: 0.22923813377952493
[I 2024-04-14 21:57:18,574] Trial 57 finished with value: 0.23596621779736537 and parameters: {'subsample': 0.13165605060453595, 'dropout_rate': 0.5678085837943805, 'n_estimators': 15, 'learning_rate': 0.014955776210569753}. Best is trial 22 with value: 0.2359281786560819.
Fold 1 IBS: 0.24628949870709554
Fold 2 IBS: 0.23031437654668757
Fold 3 IBS: 0.22884653819375786
Fold 4 IBS: 0.2458696334873806
Fold 5 IBS: 0.22922612427093866
[I 2024-04-14 21:57:24,397] Trial 58 finished with value: 0.23610923424117206 and parameters: {'subsample': 0.9550218144832968, 'dropout_rate': 0.6272447801163561, 'n_estimators': 45, 'learning_rate': 0.009156932045437691}. Best is trial 22 with value: 0.2359281786560819.
Fold 1 IBS: 0.28328758722421493
Fold 2 IBS: 0.2394754061855135
Fold 3 IBS: 0.24015286871265207
Fold 4 IBS: 0.37157961788423793
Fold 5 IBS: 0.28334915962344304
[I 2024-04-14 21:57:54,680] Trial 59 finished with value: 0.28356892792

Fold 5 IBS: 0.22930085302882294
[I 2024-04-14 21:59:05,742] Trial 76 finished with value: 0.23598931888925131 and parameters: {'subsample': 0.8768270122303635, 'dropout_rate': 0.7823662573901398, 'n_estimators': 17, 'learning_rate': 0.012589096150523053}. Best is trial 72 with value: 0.23592780329773116.
Fold 1 IBS: 0.24602578706773542
Fold 2 IBS: 0.22971738000994765
Fold 3 IBS: 0.22880853470293033
Fold 4 IBS: 0.24732034248158946
Fold 5 IBS: 0.22918171521875855
[I 2024-04-14 21:59:07,014] Trial 77 finished with value: 0.23621075189619228 and parameters: {'subsample': 0.9659229260875638, 'dropout_rate': 0.7381012419968711, 'n_estimators': 11, 'learning_rate': 0.049658881246843596}. Best is trial 72 with value: 0.23592780329773116.
Fold 1 IBS: 0.24552879084745355
Fold 2 IBS: 0.2285216831547308
Fold 3 IBS: 0.22875495950380775
Fold 4 IBS: 0.2512936333683752
Fold 5 IBS: 0.2291099378899557
[I 2024-04-14 21:59:10,577] Trial 78 finished with value: 0.23664180095286458 and parameters: {'subsamp

Fold 5 IBS: 0.22928162319464118
[I 2024-04-14 21:59:59,644] Trial 95 finished with value: 0.2360473782649713 and parameters: {'subsample': 0.9263734948124571, 'dropout_rate': 0.7166910143860685, 'n_estimators': 26, 'learning_rate': 0.012609107231273231}. Best is trial 81 with value: 0.23592753810805128.
Fold 1 IBS: 0.24646324144717863
Fold 2 IBS: 0.23065220891764882
Fold 3 IBS: 0.2288695374800145
Fold 4 IBS: 0.2450357850312088
Fold 5 IBS: 0.22926607450001513
[I 2024-04-14 22:00:04,672] Trial 96 finished with value: 0.2360573694752132 and parameters: {'subsample': 0.9009724346080058, 'dropout_rate': 0.7497063849503668, 'n_estimators': 56, 'learning_rate': 0.0059409799616813586}. Best is trial 81 with value: 0.23592753810805128.
Fold 1 IBS: 0.24696794468000824
Fold 2 IBS: 0.23157500045630858
Fold 3 IBS: 0.22894051312961808
Fold 4 IBS: 0.24292659425614868
Fold 5 IBS: 0.22934542332213265
[I 2024-04-14 22:00:06,328] Trial 97 finished with value: 0.23595109516884322 and parameters: {'subsamp

In [73]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [74]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.578
train_ibs:  0.236


#### Test

In [75]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [76]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.944388489444451,
                                              learning_rate=0.0413681807904896,
                                              n_estimators=38, random_state=123,
                                              subsample=0.30359910909282395)

C-index score: 0.597


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.3796513018058376,
                                              learning_rate=0.003225987542685412,
                                              n_estimators=1, random_state=123,
                                              subsample=0.9053231560128981)

IBS: 0.229


In [77]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [78]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
ExtraSurvivalTrees,0.859,1.0
Randomsurvivalforest,0.826,2.0
GradientBoosting,0.686,3.0
CoxElastic,0.644,4.0
CoxRidge,0.596,5.0
ComponentwiseGradientBoosting,0.578,6.0
CoxLasso,0.468,7.0


In [79]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.213,1.0
ExtraSurvivalTrees,0.217,2.0
GradientBoosting,0.231,3.0
CoxElastic,0.234,4.0
CoxRidge,0.236,5.5
ComponentwiseGradientBoosting,0.236,5.5
CoxLasso,0.433,7.0


In [80]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
ComponentwiseGradientBoosting,0.597,1.0
CoxElastic,0.532,2.0
CoxLasso,0.522,3.0
GradientBoosting,0.518,4.0
Randomsurvivalforest,0.516,5.0
CoxRidge,0.510,6.0
ExtraSurvivalTrees,0.506,7.0


In [81]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
CoxElastic,0.228,1.0
CoxRidge,0.229,2.5
ComponentwiseGradientBoosting,0.229,2.5
GradientBoosting,0.230,4.0
ExtraSurvivalTrees,0.245,5.0
Randomsurvivalforest,0.258,6.0
CoxLasso,0.423,7.0


In [86]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d2/dfs/minmax/no_selection/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d2_dfs_minmax_no_selection_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [87]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-16
